# Construct close-election RD sample (vote-margin)
This notebook defines market-oriented winners, constructs signed vote-margin
running variables, and links pre/post-election incumbency for the close-election design.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.elections_parlgov import (
    compute_bloc_ideology_distance,
    compute_cabinet_ideology,
    compute_market_seat_shares,
    compute_market_vote_shares,
    compute_top2_margin_by_bloc,
    select_post_election_cabinets,
)
from src.ideology import (
    build_ideology_bundle,
    classify_market_ideology,
    classify_market_lr,
    normalize_party_name,
    prepare_vparty_positions,
)
from src.paths import ANALYSIS_DIR, CLEAN_DIR, INTERMEDIATE_DIR, PAPER_LOGS_DIR, RAW_DIR
from src.qc import assert_unique_key
from src.viz_style import set_style

In [2]:
set_style()

panel_path = CLEAN_DIR / "panel_annual_atlas.parquet"
if not panel_path.exists():
    raise FileNotFoundError("Missing annual panel. Run 03b_build_annual_panel first.")

panel = pd.read_parquet(panel_path)

In [3]:
# Source selection (primary: CLEA if present, fallback: ParlGov)
clea_elections_path = INTERMEDIATE_DIR / "clea_elections.parquet"
clea_results_path = INTERMEDIATE_DIR / "clea_results.parquet"

parlgov_elections_path = INTERMEDIATE_DIR / "parlgov_elections.parquet"
parlgov_results_path = INTERMEDIATE_DIR / "parlgov_election_results.parquet"
parlgov_cabinets_path = INTERMEDIATE_DIR / "parlgov_cabinets.parquet"
parlgov_cabinet_parties_path = INTERMEDIATE_DIR / "parlgov_cabinet_parties.parquet"
parlgov_parties_path = INTERMEDIATE_DIR / "parlgov_parties.parquet"

use_clea = clea_elections_path.exists() and clea_results_path.exists()
source_label = "clea" if use_clea else "parlgov"

In [4]:
START_YEAR = 2000
max_year = int(panel["year"].max()) - 3

In [5]:
# Helper: compute top-2 margin from national vote shares

def compute_top2_margin(results: pd.DataFrame, vote_share_col: str = "vote_share") -> pd.DataFrame:
    frame = results.dropna(subset=[vote_share_col]).copy()
    frame["vote_share_raw"] = frame[vote_share_col].astype(float)
    scale = 100.0 if frame["vote_share_raw"].max() > 1.5 else 1.0
    frame["vote_share_frac"] = frame["vote_share_raw"] / scale
    frame = frame.sort_values(["election_id", "vote_share_frac"], ascending=[True, False])
    top2 = frame.groupby("election_id", as_index=False).head(2).copy()
    top2["rank"] = top2.groupby("election_id").cumcount() + 1

    share = (
        top2.pivot_table(index="election_id", columns="rank", values="vote_share_frac", aggfunc="first")
        .rename(columns={1: "top1_share", 2: "top2_share"})
        .reset_index()
    )
    party = (
        top2.pivot_table(index="election_id", columns="rank", values="party_id", aggfunc="first")
        .rename(columns={1: "top1_party_id", 2: "top2_party_id"})
        .reset_index()
    )
    party_name = (
        top2.pivot_table(index="election_id", columns="rank", values="party_name", aggfunc="first")
        .rename(columns={1: "top1_party_name", 2: "top2_party_name"})
        .reset_index()
    )

    out = share.merge(party, on="election_id", how="left")
    out = out.merge(party_name, on="election_id", how="left")
    out["top2_margin"] = out["top1_share"] - out["top2_share"]
    out["top2_margin_abs"] = out["top2_margin"].abs()
    return out

In [6]:
if use_clea:
    elections = pd.read_parquet(clea_elections_path)
    results = pd.read_parquet(clea_results_path)

    elections = elections[elections["election_year"].notna()].copy()
    elections["election_year"] = elections["election_year"].astype(int)
    elections = elections[(elections["election_year"] >= START_YEAR) & (elections["election_year"] <= max_year)]

    top2_margin = compute_top2_margin(results)
    elections = elections.merge(top2_margin, on="election_id", how="left")

    # Ideology mapping (V-Party primary; DPI fallback for incumbency)
    dpi_path = RAW_DIR / "dpi" / "dpi2012.xls"
    ideology = build_ideology_bundle(dpi_path, RAW_DIR / "vparty")

    vparty_path = RAW_DIR / "vparty" / "CPD_V-Party_CSV_v2" / "V-Dem-CPD-Party-V2.csv"
    vparty_map = prepare_vparty_positions(vparty_path) if vparty_path.exists() else None

    results = results.copy()
    results["party_name_norm"] = results["party_name"].apply(normalize_party_name)

    if vparty_map is not None and not vparty_map.empty:
        results = results.merge(
            vparty_map,
            left_on=["iso3c", "election_year", "party_name_norm"],
            right_on=["iso3c", "year", "party_name_norm"],
            how="left",
        )
        results["market_party"] = results["ideology_lr"].apply(classify_market_lr)

        vote_totals = results.groupby("election_id")["votes"].sum().rename("vote_total")
        market_votes = (
            results.loc[results["market_party"] == 1]
            .groupby("election_id")["votes"]
            .sum()
            .rename("vote_market")
        )
        elections = elections.merge(vote_totals, on="election_id", how="left")
        elections = elections.merge(market_votes, on="election_id", how="left")
        elections["vote_share_market"] = elections["vote_market"] / elections["vote_total"].replace(0, np.nan)

        elections["winner_market"] = (elections["vote_share_market"] > 0.5).astype(float)
        elections["running_var_vote"] = elections["vote_share_market"] - 0.5
        elections["running_var_vote_margin"] = 2 * elections["vote_share_market"] - 1
    else:
        elections["vote_share_market"] = np.nan
        elections["winner_market"] = np.nan
        elections["running_var_vote"] = np.nan
        elections["running_var_vote_margin"] = np.nan

    # DPI incumbency fallback
    if ideology.dpi_exec is not None:
        dpi = ideology.dpi_exec.copy()
        dpi["market_exec"] = dpi["exec_rlc"].apply(classify_market_ideology)
        dpi["market_gov1"] = dpi["gov1_rlc"].apply(classify_market_ideology)
        dpi["market_best"] = dpi["market_exec"].combine_first(dpi["market_gov1"])

        incumbent = dpi[["iso3c", "year", "market_best"]].copy()
        incumbent["election_year"] = incumbent["year"] + 1
        incumbent = incumbent.rename(columns={"market_best": "incumbent_market"})
        elections = elections.merge(
            incumbent[["iso3c", "election_year", "incumbent_market"]],
            on=["iso3c", "election_year"],
            how="left",
        )
    else:
        elections["incumbent_market"] = np.nan

    # Signed running variable: top-2 margin signed by winner ideology (fallback)
    elections["running_var_top2"] = elections["top2_margin"]
    elections["running_var_vote_alt"] = np.nan
    elections["running_var_seat"] = np.nan
    elections["running_var_seat_alt"] = np.nan
    elections["vote_share_market_alt"] = np.nan

    if elections["running_var_vote"].notna().sum() == 0 and elections["winner_market"].notna().sum() > 0:
        elections["running_var_vote"] = elections["top2_margin"] * (2 * elections["winner_market"] - 1)
        elections["running_var_vote_margin"] = elections["running_var_vote"]

    elections["market_majority_vote"] = (elections["running_var_vote"] > 0).astype(float)
    elections["market_majority_vote_alt"] = np.nan
    elections["market_majority_seat"] = np.nan
    elections["market_majority_top2"] = (elections["running_var_top2"] > 0).astype(float)

    elections["market_switch"] = np.where(
        elections["winner_market"].notna() & elections["incumbent_market"].notna(),
        (elections["winner_market"] != elections["incumbent_market"]).astype(float),
        np.nan,
    )
else:
    for path in [
        parlgov_elections_path,
        parlgov_results_path,
        parlgov_cabinets_path,
        parlgov_cabinet_parties_path,
        parlgov_parties_path,
    ]:
        if not path.exists():
            raise FileNotFoundError(f"Missing ParlGov output: {path}")

    elections = pd.read_parquet(parlgov_elections_path)
    results = pd.read_parquet(parlgov_results_path)
    cabinets = pd.read_parquet(parlgov_cabinets_path)
    cabinet_parties = pd.read_parquet(parlgov_cabinet_parties_path)
    parties = pd.read_parquet(parlgov_parties_path)

    MARKET_THRESHOLD = 5.0
    ALT_THRESHOLD = 6.0

    elections = elections[elections["election_type"].str.contains("Parliament", case=False, na=False)].copy()
    elections = elections[elections["year"].notna()].copy()
    elections["year"] = elections["year"].astype(int)
    elections = elections[(elections["year"] >= START_YEAR) & (elections["year"] <= max_year)].copy()

    results = results.dropna(subset=["seats"]).copy()

    seat_share_main = compute_market_seat_shares(results, parties, threshold=MARKET_THRESHOLD)
    seat_share_alt = compute_market_seat_shares(results, parties, threshold=ALT_THRESHOLD).rename(
        columns={
            "seat_market": "seat_market_alt",
            "seat_share_market": "seat_share_market_alt",
        }
    )

    vote_share_main = compute_market_vote_shares(results, parties, threshold=MARKET_THRESHOLD)
    vote_share_alt = compute_market_vote_shares(results, parties, threshold=ALT_THRESHOLD).rename(
        columns={
            "vote_share_market_raw": "vote_share_market_raw_alt",
            "vote_share_market": "vote_share_market_alt",
        }
    )

    ideology_distance = compute_bloc_ideology_distance(results, parties, threshold=MARKET_THRESHOLD)
    top2_margin = compute_top2_margin_by_bloc(results, parties, threshold=MARKET_THRESHOLD)

    elections = elections.merge(seat_share_main, on="election_id", how="left")
    elections = elections.merge(
        seat_share_alt[["election_id", "seat_market_alt", "seat_share_market_alt"]],
        on="election_id",
        how="left",
    )
    elections = elections.merge(vote_share_main, on="election_id", how="left")
    elections = elections.merge(
        vote_share_alt[["election_id", "vote_share_market_alt"]],
        on="election_id",
        how="left",
    )
    elections = elections.merge(
        ideology_distance[["election_id", "lr_market", "lr_nonmarket", "lr_distance", "lr_distance_abs"]],
        on="election_id",
        how="left",
    )
    elections = elections.merge(
        top2_margin[
            [
                "election_id",
                "top_market_share",
                "top_nonmarket_share",
                "top2_margin",
                "top2_margin_abs",
                "winner_market_top2",
            ]
        ],
        on="election_id",
        how="left",
    )

    # Running variables
    elections["running_var_seat"] = elections["seat_share_market"] - 0.5
    elections["running_var_seat_alt"] = elections["seat_share_market_alt"] - 0.5
    elections["running_var_vote"] = elections["vote_share_market"] - 0.5
    elections["running_var_vote_margin"] = 2 * elections["vote_share_market"] - 1
    elections["running_var_vote_alt"] = elections["vote_share_market_alt"] - 0.5
    elections["running_var_top2"] = elections["top2_margin"]

    # Treatment indicators
    elections["market_majority_seat"] = (elections["running_var_seat"] > 0).astype(int)
    elections["market_majority_vote"] = (elections["running_var_vote"] > 0).astype(int)
    elections["market_majority_vote_alt"] = (elections["running_var_vote_alt"] > 0).astype(int)
    elections["market_majority_top2"] = (elections["running_var_top2"] > 0).astype(int)

    # Cabinet ideology and incumbency
    cabinet_summary = compute_cabinet_ideology(
        cabinets,
        cabinet_parties,
        parties,
        results,
        threshold=MARKET_THRESHOLD,
    )

    post_cabinets = select_post_election_cabinets(cabinets)
    post_cabinets = post_cabinets.merge(
        cabinet_summary,
        left_on="post_cabinet_id",
        right_on="cabinet_id",
        how="left",
    )
    post_cabinets = post_cabinets.rename(
        columns={
            "cabinet_lr": "post_cabinet_lr",
            "cabinet_market_share": "post_cabinet_market_share",
            "cabinet_party_count": "post_cabinet_party_count",
        }
    )
    post_cabinets["winner_market"] = (post_cabinets["post_cabinet_lr"] >= MARKET_THRESHOLD).astype(float)

    incumbent = post_cabinets.merge(
        cabinet_summary,
        left_on="previous_cabinet_id",
        right_on="cabinet_id",
        how="left",
        suffixes=("", "_incumbent"),
    )
    incumbent = incumbent.rename(
        columns={
            "cabinet_lr": "incumbent_cabinet_lr",
            "cabinet_market_share": "incumbent_cabinet_market_share",
            "cabinet_party_count": "incumbent_cabinet_party_count",
        }
    )
    incumbent["incumbent_market"] = (incumbent["incumbent_cabinet_lr"] >= MARKET_THRESHOLD).astype(float)

    cabinet_cols = [
        "election_id",
        "post_cabinet_id",
        "previous_cabinet_id",
        "post_cabinet_lr",
        "post_cabinet_market_share",
        "post_cabinet_party_count",
        "winner_market",
        "incumbent_cabinet_lr",
        "incumbent_cabinet_market_share",
        "incumbent_cabinet_party_count",
        "incumbent_market",
    ]

    elections = elections.merge(incumbent[cabinet_cols], on="election_id", how="left")
    elections["market_switch"] = (elections["winner_market"] != elections["incumbent_market"]).astype(float)

In [7]:
# Keep one election per country-year
if "election_date" in elections.columns:
    elections = elections.sort_values("election_date")
else:
    elections = elections.sort_values(["iso3c", "election_year", "election_id"])

if "year" in elections.columns:
    elections = elections.rename(columns={"year": "election_year"})

# Align to EFW coverage
efw_countries = panel["iso3c"].dropna().unique().tolist()

if "iso3c" not in elections.columns:
    raise ValueError("Elections data must include iso3c codes.")

elections = elections[elections["iso3c"].isin(efw_countries)].copy()

# Select last election per country-year
if "election_year" not in elections.columns:
    raise ValueError("Elections data must include election_year.")

elections = elections.groupby(["iso3c", "election_year"], as_index=False).tail(1).reset_index(drop=True)

In [8]:
# Finalize events
assert_unique_key(elections, ["iso3c", "election_year"])

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
sample_path = ANALYSIS_DIR / "close_elections_vote_margin.parquet"
legacy_path = ANALYSIS_DIR / "close_elections_sample.parquet"

elections.to_parquet(sample_path, index=False)
elections.to_parquet(legacy_path, index=False)

coverage = pd.DataFrame(
    {
        "rows": [len(elections)],
        "countries": [elections["iso3c"].nunique()],
        "min_year": [elections["election_year"].min()],
        "max_year": [elections["election_year"].max()],
        "share_market_majority_vote": [elections["market_majority_vote"].mean()],
        "share_market_majority_seat": [
            elections["market_majority_seat"].mean() if "market_majority_seat" in elections.columns else np.nan
        ],
    }
)

display(coverage.style.set_caption("Close-election sample summary"))
display(elections.head(5).style.set_caption("Sample rows"))

running = elections["running_var_vote"].dropna()
heaping_share = (running.round(2) == running).mean() if not running.empty else np.nan
missing_vote_share = elections["vote_share_market"].isna().mean() if "vote_share_market" in elections.columns else np.nan
missing_top2 = elections["running_var_top2"].isna().mean() if "running_var_top2" in elections.columns else np.nan

meta = {
    "build": {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "pipeline_version": "rd-iv-vote-margin-v2",
    },
    "definitions": {
        "source": source_label,
        "running_variable": (
            "vote_share_market - 0.5 (V-Party bloc)"
            if use_clea and elections["vote_share_market"].notna().any()
            else ("signed top-2 vote margin (winner ideology)" if use_clea else "vote_share_market - 0.5")
        ),
    },
    "inputs": {
        "panel_annual": str(panel_path),
        "clea_elections": str(clea_elections_path) if use_clea else None,
        "clea_results": str(clea_results_path) if use_clea else None,
        "parlgov_elections": str(parlgov_elections_path) if not use_clea else None,
        "parlgov_results": str(parlgov_results_path) if not use_clea else None,
    },
    "summary": {
        **coverage.to_dict(orient="records")[0],
        "missing_vote_share": missing_vote_share,
        "missing_top2_margin": missing_top2,
    },
    "qc": {"running_var_vote_heaping_share": heaping_share},
}

meta_path = PAPER_LOGS_DIR / "close_elections_vote_margin_metadata.json"
meta_path.parent.mkdir(parents=True, exist_ok=True)
meta_path.write_text(json.dumps(meta, indent=2))

,rows,countries,min_year,max_year,share_market_majority_vote,share_market_majority_seat
0,295,36,2000,2021,0.657627,0.677966


,election_id,type_id,country_id,date,first_round_election_id,early,wikipedia,seats_total,electorate,votes_cast,votes_valid,data_source,description,comment,previous_parliament_election_id,previous_ep_election_id,previous_cabinet_id_x,old_countryID,old_parlID,country_name,iso3c,iso_numeric,election_date,election_year,election_type,seat_total,seat_market,seat_share_market,seat_market_alt,seat_share_market_alt,vote_share_total,vote_share_market_raw,vote_share_market,vote_share_market_alt,lr_market,lr_nonmarket,lr_distance,lr_distance_abs,top_market_share,top_nonmarket_share,top2_margin,top2_margin_abs,winner_market_top2,running_var_seat,running_var_seat_alt,running_var_vote,running_var_vote_margin,running_var_vote_alt,running_var_top2,market_majority_seat,market_majority_vote,market_majority_vote_alt,market_majority_top2,post_cabinet_id,previous_cabinet_id_y,post_cabinet_lr,post_cabinet_market_share,post_cabinet_party_count,winner_market,incumbent_cabinet_lr,incumbent_cabinet_market_share,incumbent_cabinet_party_count,incumbent_market,market_switch
0,797,13,62,2000-01-03,nan,0,"http://en.wikipedia.org/wiki/Croatian_parliamentary_election,_2000",151,3686378.000000,2821020.000000,2774275.000000,no-2010,None,None,nan,nan,nan,nan,nan,Croatia,HRV,191,2000-01-03 00:00:00,2000,Parliamentary election,151.000000,92.000000,0.609272,92.000000,0.609272,92.600000,49.200000,0.531317,0.516199,7.001601,3.245600,3.756001,3.756001,0.244000,0.408000,-0.164000,0.164000,0.000000,0.109272,0.109272,0.031317,0.062635,0.016199,-0.164000,1,1,1,0,1049.000000,1049.000000,4.777405,0.440860,6.000000,0.000000,4.777405,0.440860,6.000000,0.000000,0.000000
1,687,13,27,2000-03-12,nan,0,"http://en.wikipedia.org/wiki/Spanish_general_election,_2000",350,33969640.000000,23339490.000000,22814467.000000,epp,None,None,99.000000,585.000000,370.000000,724.000000,20000.000000,Spain,ESP,724,2000-03-12 00:00:00,2000,Parliamentary election,350.000000,209.000000,0.597143,205.000000,0.585714,97.080000,52.130000,0.536980,0.525752,7.412676,3.493823,3.918853,3.918853,0.452400,0.347100,0.105300,0.105300,1.000000,0.097143,0.085714,0.036980,0.073960,0.025752,0.105300,1,1,1,1,309.000000,370.000000,7.596900,1.000000,1.000000,1.000000,7.596900,1.000000,1.000000,1.000000,0.000000
2,475,13,41,2000-04-09,nan,0,"http://en.wikipedia.org/wiki/Greek_legislative_election,_2000",300,9372541.000000,7026527.000000,6868011.000000,mig,None,None,399.000000,482.000000,33.000000,300.000000,20000.000000,Greece,GRC,300,2000-04-09 00:00:00,2000,Parliamentary election,300.000000,125.000000,0.416667,125.000000,0.416667,97.940000,42.740000,0.436390,0.436390,6.736500,3.965163,2.771337,2.771337,0.427400,0.437900,-0.010500,0.010500,0.000000,-0.083333,-0.083333,-0.063610,-0.127221,-0.063610,-0.010500,0,0,0,0,190.000000,33.000000,4.496800,0.000000,1.000000,0.000000,4.496800,0.000000,1.000000,0.000000,0.000000
3,237,13,5,2000-06-25,nan,1,"http://en.wikipedia.org/wiki/Japanese_general_election,_2000",480,100492328.000000,62757828.000000,59844601.000000,parline,None,None,628.000000,nan,211.000000,392.000000,20000.000000,Japan,JPN,392,2000-06-25 00:00:00,2000,Parliamentary election,480.000000,421.000000,0.877083,263.000000,0.547917,99.820000,78.980000,0.791224,0.409036,6.744925,1.823702,4.921223,4.921223,0.283100,0.112300,0.170800,0.170800,1.000000,0.377083,0.047917,0.291224,0.582448,-0.090964,0.170800,1,1,0,1,163.000000,211.000000,7.764001,1.000000,3.000000,1.000000,8.016504,1.000000,3.000000,1.000000,0.000000
4,59,13,15,2000-10-08,nan,0,"http://en.wikipedia.org/wiki/Lithuanian_parliamentary_election,_2000",141,2626321.000000,1539743.000000,1471247.000000,vrk,None,2000 election with plurality instead of majority run-off for majority tier of parallel system,566.000000,nan,735.000000,440.000000,20000.000000,Lithuania,LTU,440,2000-10-08 00:00:00,2000,Parliamentary election,141.000000,49.000000,0.347518,49.000000,0.347518,98.640000,37.560000,0.380779,0.380779,7.371472,3.686107,3.685364,3.685364,0.172500,0.310800,-0.138300

1003

## Interpretation
The close-election sample uses a signed vote-margin running variable that is positive
for more-market winners and negative for less-market winners. CLEA is the primary
source when available; ParlGov is used only as a fallback.